In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score, precision_score,
    recall_score, matthews_corrcoef,
)

from sklearn.metrics import confusion_matrix
from scipy.stats import bootstrap, beta

In [2]:
# connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Load results

In [3]:
def load_results(file_path, model_name, task):

    df = pd.read_csv(file_path)
    df = df.dropna(subset=["pred"])

    test_files = {
        "anx": "anxiety_test.csv",
        "dep": "depression_test.csv",
        "ed": "ed_test.csv",
        "multi": "multiclass_test.csv"
    }

    test_df = pd.read_csv(
        "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/BERT/"
        + test_files[task]
    )

    # ========================================
    # Filtering using the same test as BERT
    # ========================================

    if task == "multi":

        test_ids = (
            test_df["user_id"].astype(str)
            + "_"
            + test_df["trastorno"].astype(str)
        )

        current_ids = (
            df["user_id"].astype(str)
            + "_"
            + df["trastorno"].astype(str)
        )

        df = df[current_ids.isin(test_ids)]

    else:

        df = df[
            df["user_id"].isin(test_df["user_id"])
        ]

    # ========================================
    # Construction of sample_id
    # ========================================

    if task == "multi":

        true_col = "label_mc"

        sample_id = (
            df["user_id"].astype(str)
            + "_"
            + df["trastorno"].astype(str)
        )

    else:

        true_col = "bs"
        sample_id = df["user_id"].astype(str)

    return pd.DataFrame({
        "sample_id": sample_id,
        "user_id": df["user_id"],
        "model": model_name,
        "task": task,
        "y_true": df[true_col],
        "y_pred": df["pred"].astype(int)
    })

In [4]:
base = "/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/"
tasks = ["anx", "dep", "ed", "multi"]

bert_models = [
    "distilbert-base-spanish-uncased",
    "roberta-large-bne",
    "mdeberta-v3-base"
    # "longformer-base-4096"
]
open_source_llms = [
    "llama-3.1-8b",
    "gpt-oss-20b",
    "deepseek-chat"
]
proprietary_llms = [
    "gpt-5.2",
    "gpt-5-mini",
    "claude-sonnet-4-6",
    "claude-haiku-4.5",
    "gemini-3.1-pro-preview",
    "gemini-3-flash-preview"
]

llm_models = open_source_llms + proprietary_llms
models = bert_models + llm_models

dfs = []

# Load results for each task and model
for task in tasks:
  for model in models:
    results = load_results(base + f"{model}_{task}.csv", model, task)
    dfs.append(results)

# Combine them into a single DataFrame
data = pd.concat(dfs, ignore_index=True)

In [5]:
data

,sample_id,user_id,model,task,y_true,y_pred
0,subject140,subject140,distilbert-base-spanish-uncased,anx,1,1
1,subject166,subject166,distilbert-base-spanish-uncased,anx,1,1
2,subject372,subject372,distilbert-base-spanish-uncased,anx,1,1
3,subject365,subject365,distilbert-base-spanish-uncased,anx,1,1
4,subject404,subject404,distilbert-base-spanish-uncased,anx,0,1
...,...,...,...,...,...,...
4811,subject68_ed,subject68,gemini-3-flash-preview,multi,3,3
4812,subject8_ed,subject8,gemini-3-flash-preview,multi,3,3
4813,subject83_ed,subject83,gemini-3-flash-preview,multi,0,2
4814,subject95_ed,subject95,gemini-3-flash-preview,multi,3,3


In [6]:
# Number of samples per set and model
for task in tasks:
    print("\n", task)

    print(
        data[data["task"] == task]
        .groupby("model")
        .size()
        .sort_values()
        .head()
    )


 anx
model
claude-haiku-4.5                   75
claude-sonnet-4-6                  75
deepseek-chat                      75
distilbert-base-spanish-uncased    75
gemini-3-flash-preview             75
dtype: int64

 dep
model
gemini-3.1-pro-preview             73
gemini-3-flash-preview             73
claude-haiku-4.5                   75
claude-sonnet-4-6                  75
distilbert-base-spanish-uncased    75
dtype: int64

 ed
model
claude-haiku-4.5                   51
claude-sonnet-4-6                  51
deepseek-chat                      51
distilbert-base-spanish-uncased    51
gemini-3-flash-preview             51
dtype: int64

 multi
model
gemini-3.1-pro-preview             199
gemini-3-flash-preview             199
claude-haiku-4.5                   201
claude-sonnet-4-6                  201
distilbert-base-spanish-uncased    201
dtype: int64


In [ ]:
# Save data to use in the rest of the notebooks
CACHE_DIR = base + "metricas"
data.to_parquet(f"{CACHE_DIR}/all_predictions.parquet")

# 2. Main metrics table

This block builds the main results table for each task:
- General classification metrics
- Bootstrap confidence intervals (95%)
- Block Rate (only LLMs)
- TARa@5 (only LLMs)

It is valid for both binary and multiclass classification.


## 2.1. General metrics

In [7]:
def compute_metrics(y_true, y_pred):
    """
    Calculates the main evaluation metrics.

    Used:
    - F1 macro: main metric (especially important with imbalance)
    - Precision macro
    - Recall macro
    - MCC (Matthews Correlation Coefficient): robust metric for imbalanced classification
    """
    metrics = {}

    metrics["f1_macro"] = f1_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["precision"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["recall"] = recall_score(y_true, y_pred, average="macro", zero_division=0)
    metrics["mcc"] = matthews_corrcoef(y_true, y_pred)

    return metrics

## 2.2. Bootstrap confidence intervals (95%)

In [8]:
def bootstrap_ci(y_true, y_pred, n_bootstrap=1000):
    """
    Calculates the 95% confidence interval using BCa bootstrap
    (Bias-Corrected and Accelerated).
    An independent BCa bootstrap is calculated for each metric.
    """

    ci = {}

    # =========================================================
    # Define a function per metric
    # =========================================================

    metric_functions = {
        "f1_macro": lambda yt, yp: f1_score(
            yt,
            yp,
            average="macro",
            zero_division=0
        ),

        "precision": lambda yt, yp: precision_score(
            yt,
            yp,
            average="macro",
            zero_division=0
        ),

        "recall": lambda yt, yp: recall_score(
            yt,
            yp,
            average="macro",
            zero_division=0
        ),

        "mcc": lambda yt, yp: matthews_corrcoef(
            yt,
            yp
        )
    }

    # =========================================================
    # Calculate CI for each metric
    # =========================================================

    for metric_name, metric_fn in metric_functions.items():

        result = bootstrap(
            (y_true.values, y_pred.values),
            metric_fn,
            paired=True, # same resampling for y_true and y_pred
            vectorized=False,
            confidence_level=0.95,
            n_resamples=n_bootstrap,
            method="BCa",
            random_state=42
        )

        lower = result.confidence_interval.low
        upper = result.confidence_interval.high

        # If BCa returns NaN → fallback percentile
        # Applied in cases where predictions are almost identical
        # and BCa throws DegenerateDataWarning
        if np.isnan(lower) or np.isnan(upper):

            values = []

            for _ in range(n_bootstrap):
                idx = np.random.choice(len(y_true), len(y_true), replace=True)

                yt = y_true.iloc[idx]
                yp = y_pred.iloc[idx]

                values.append(metric_fn(yt, yp))

            lower = np.percentile(values, 2.5)
            upper = np.percentile(values, 97.5)

        ci[metric_name] = (lower, upper)

    return ci

## 2.3. LLM-specific metrics

### 2.3.1. Block rate (only LLMs)

In [9]:
def compute_block_rate(file_path):
    """
    Calculates the percentage of blocked or failed model responses.
    Measured as: pred == NaN. Only applies to LLMs.
    """
    df = pd.read_csv(file_path)
    return df["pred"].isna().mean()

### 2.3.2. TARa@5

In [10]:
def load_tara_df(model, task):
    """
    Loads the necessary data to calculate TARa@5.
    """

    # Complete dataset (original prediction) and rename "pred" column to "pred_run1"
    df_full = pd.read_csv(base + f"{model}_{task}.csv")
    df_full = df_full[["user_id", "pred"]].rename(columns={"pred": "pred_run1"})

    # Subset used for TARa@5 (contains "pred_run2", ..., "pred_run5")
    df_tara = pd.read_csv(base + f"{model}_{task}_tara5.csv")

    # Merge both by user_id to add pred_run1 to df_tara
    df_tara = df_tara.merge(df_full, on="user_id", how="left")

    return df_tara

In [ ]:
def compute_tara(df_tara):
    """
    Calculates TARa@5: For each example, the 5 predictions are compared and
    the degree of agreement with respect to the mode is calculated. The final result is the mean
    agreement over all examples.

    Empty predictions (NaN) are considered disagreement,
    as they represent generation failures or model blockage.

    Example:
    [1, 1, 1, NaN, 0]
    → mode = 1
    → agreement = 3/5 = 0.60
    """

    pred_cols = [
        "pred_run1",
        "pred_run2",
        "pred_run3",
        "pred_run4",
        "pred_run5"
    ]

    agreements = []

    for _, row in df_tara.iterrows():

        # Always extract the 5 predictions (including NaN)
        preds = row[pred_cols]

        # If all are NaN, mode cannot be calculated
        if preds.dropna().empty:
            continue

        # The mode is calculated only on valid predictions
        majority_class = preds.dropna().mode()[0]

        # NaN counts as disagreement because != majority_class
        agreement = (preds == majority_class).mean()

        agreements.append(agreement)

    return np.mean(agreements) if agreements else np.nan

In [ ]:
def tara_credible_interval(
    df_tara,
    alpha_prior=1,
    beta_prior=1
):
    """
    95% credible interval for TARa@5 using a
    Beta-Binomial model.

    Each subject provides 5 trials (pred_run1...pred_run5).
    """

    pred_cols = [
        "pred_run1",
        "pred_run2",
        "pred_run3",
        "pred_run4",
        "pred_run5"
    ]

    agreements = []

    for _, row in df_tara.iterrows():

        preds = row[pred_cols]

        if preds.dropna().empty:
            continue

        majority = preds.dropna().mode()[0]

        k = (preds == majority).sum()

        agreements.append(k)

    k_total = np.sum(agreements)

    n_total = 5 * len(agreements)

    a_post = alpha_prior + k_total
    b_post = beta_prior + (n_total - k_total)

    return (
        beta.ppf(0.025, a_post, b_post),
        beta.ppf(0.975, a_post, b_post)
    )

## 2.4. Build final table with all metrics

In [ ]:
def metrics_with_ci(df, task):
    """
    Builds the final metrics table for a task using the previous functions.
    """
    # Results formatted to display in table with CIs, for example: 0.636 [0.478, 0.835]
    results = []
    # Same as results but saves CIs separately, useful for later statistics
    results_raw = []

    # Iterate over each model
    for model in df["model"].unique():
        subset = df[df["model"] == model]

        y_true = subset["y_true"]
        y_pred = subset["y_pred"]

        # General metrics
        base_metrics = compute_metrics(y_true, y_pred)

        # 95% BCa bootstrap CI
        ci = bootstrap_ci(y_true, y_pred)

        # Save each metric with its CI
        row = {"model": model}
        row_raw = {"model": model}
        for metric in base_metrics:
            low, high = ci[metric]

            row[metric] = f"{base_metrics[metric]:.3f} [{low:.3f}, {high:.3f}]"
            row_raw[f"{metric}"] = base_metrics[metric]
            row_raw[f"{metric}_low"] = low
            row_raw[f"{metric}_high"] = high

        # Block rate
        if model in llm_models:
            file_path = base + f"{model}_{task}.csv"
            block_rate = compute_block_rate(file_path)

            row["block_rate"] = f"{block_rate:.3f}"
            row_raw["block_rate"] = block_rate
        else:
            row["block_rate"] = "-"
            row_raw["block_rate"] = np.nan

        # TARA@5
        if model in llm_models:
            df_tara = load_tara_df(model, task)

            tara = compute_tara(df_tara)
            low, high = tara_credible_interval(df_tara)

            row["tara@5"] = f"{tara:.3f} [{low:.3f}, {high:.3f}]"
            row_raw["tara@5"] = tara
            row_raw["tara@5_low"] = low
            row_raw["tara@5_high"] = high
        else:
            row["tara@5"] = "-"
            row_raw["tara@5"] = np.nan
            row_raw["tara@5_low"] = np.nan
            row_raw["tara@5_high"] = np.nan

        results.append(row)
        results_raw.append(row_raw)

    return (
        # Results formatted to display in table
        pd.DataFrame(results).set_index("model"),
        # Raw results for later analysis
        pd.DataFrame(results_raw).set_index("model")
    )

# Complete execution (per task)

In [ ]:
def display_table(task):

    df_task = data[data["task"] == task]

    table_visual, table_raw = metrics_with_ci(df_task, task)

    display(table_visual)

    return table_visual, table_raw

In [ ]:
metrics_anx, metricas_anx_raw = display_table("anx")

,f1_macro,precision,recall,mcc,block_rate,tara@5
model,,,,,,
distilbert-base-spanish-uncased,"0.650 [0.476, 0.823]","0.670 [0.458, 0.873]","0.636 [0.478, 0.835]","0.305 [-0.056, 0.644]",-,-
roberta-large-bne,"0.680 [0.529, 0.830]","0.660 [0.531, 0.811]","0.717 [0.527, 0.880]","0.373 [0.073, 0.658]",-,-
mdeberta-v3-base,"0.593 [0.460, 0.791]","0.616 [0.443, 0.897]","0.581 [0.470, 0.798]","0.194 [-0.079, 0.572]",-,-
llama-3.1-8b,"0.571 [0.464, 0.855]","0.946 [0.453, 0.993]","0.556 [0.500, 0.798]","0.315 [0.000, 0.806]",0.000,"0.980 [0.964, 0.989]"
gpt-oss-20b,"0.900 [0.728, 1.000]","0.923 [0.622, 0.993]","0.881 [0.682, 0.993]","0.803 [0.412, 0.953]",0.000,"0.988 [0.974, 0.994]"
deepseek-chat,"0.718 [0.552, 0.849]","0.690 [0.548, 0.825]","0.773 [0.548, 0.913]","0.455 [0.120, 0.701]",0.000,"0.988 [0.974, 0.994]"
gpt-5.2,"0.789 [0.609, 0.951]","0.965 [0.473, 0.986]","0.722 [0.571, 0.917]","0.643 [0.362, 0.906]",0.000,"0.998 [0.989, 1.000]"
gpt-5-mini,"0.789 [0.609, 0.951]","0.965 [0.473, 0.986]","0.722 [0.571, 0.917]","0.643 [0.362, 0.906]",0.000,"0.990 [0.977, 0.996]"
claude-sonnet-4-6,"0.889 [0.679, 1.000]","0.978 [0.942, 0.993]","0.833 [0.634, 1.000]","0.799 [0.479, 1.000]",0.000,"0.994 [0.983, 0.998]"


In [ ]:
metrics_dep, metricas_dep_raw = display_table("dep")

,f1_macro,precision,recall,mcc,block_rate,tara@5
model,,,,,,
distilbert-base-spanish-uncased,"0.797 [0.692, 0.888]","0.790 [0.686, 0.889]","0.810 [0.697, 0.892]","0.600 [0.386, 0.774]",-,-
roberta-large-bne,"0.831 [0.728, 0.915]","0.825 [0.723, 0.906]","0.860 [0.753, 0.925]","0.684 [0.483, 0.828]",-,-
mdeberta-v3-base,"0.785 [0.670, 0.869]","0.778 [0.670, 0.864]","0.800 [0.680, 0.876]","0.577 [0.350, 0.736]",-,-
llama-3.1-8b,"0.775 [0.667, 0.871]","0.805 [0.680, 0.893]","0.760 [0.650, 0.854]","0.564 [0.336, 0.737]",0.002,"0.964 [0.944, 0.977]"
gpt-oss-20b,"0.762 [0.644, 0.857]","0.784 [0.656, 0.879]","0.750 [0.628, 0.844]","0.533 [0.284, 0.708]",0.000,"0.964 [0.944, 0.977]"
deepseek-chat,"0.844 [0.740, 0.915]","0.836 [0.739, 0.906]","0.870 [0.768, 0.929]","0.705 [0.519, 0.836]",0.000,"0.988 [0.974, 0.994]"
gpt-5.2,"0.803 [0.702, 0.885]","0.806 [0.699, 0.885]","0.800 [0.694, 0.882]","0.606 [0.395, 0.761]",0.000,"0.996 [0.986, 0.999]"
gpt-5-mini,"0.809 [0.695, 0.894]","0.910 [0.855, 0.951]","0.780 [0.682, 0.870]","0.678 [0.524, 0.814]",0.000,"0.972 [0.954, 0.983]"
claude-sonnet-4-6,"0.702 [0.587, 0.817]","0.741 [0.610, 0.855]","0.690 [0.586, 0.800]","0.428 [0.198, 0.642]",0.000,"0.992 [0.980, 0.997]"


In [ ]:
metrics_ed, metricas_ed_raw = display_table("ed")

,f1_macro,precision,recall,mcc,block_rate,tara@5
model,,,,,,
distilbert-base-spanish-uncased,"0.920 [0.822, 0.980]","0.920 [0.814, 0.977]","0.920 [0.819, 0.978]","0.840 [0.629, 0.958]",-,-
roberta-large-bne,"0.940 [0.838, 0.980]","0.943 [0.859, 1.000]","0.937 [0.816, 0.983]","0.880 [0.669, 0.961]",-,-
mdeberta-v3-base,"0.816 [0.694, 0.915]","0.827 [0.696, 0.912]","0.812 [0.692, 0.912]","0.639 [0.385, 0.818]",-,-
llama-3.1-8b,"0.852 [0.718, 0.939]","0.903 [0.833, 0.959]","0.841 [0.725, 0.932]","0.741 [0.557, 0.885]",0.000,"0.952 [0.930, 0.967]"
gpt-oss-20b,"0.960 [0.863, 1.000]","0.960 [0.863, 1.000]","0.960 [0.866, 1.000]","0.920 [0.723, 1.000]",0.000,"0.952 [0.930, 0.967]"
deepseek-chat,"0.919 [0.813, 0.978]","0.927 [0.838, 0.983]","0.915 [0.804, 0.972]","0.841 [0.646, 0.957]",0.000,"0.996 [0.986, 0.999]"
gpt-5.2,"0.980 [0.899, 1.000]","0.983 [0.917, 1.000]","0.977 [0.893, 1.000]","0.961 [0.817, 1.000]",0.000,"0.992 [0.980, 0.997]"
gpt-5-mini,"0.901 [0.800, 0.961]","0.901 [0.796, 0.962]","0.908 [0.805, 0.967]","0.809 [0.610, 0.924]",0.000,"0.924 [0.897, 0.944]"
claude-sonnet-4-6,"0.960 [0.862, 1.000]","0.960 [0.859, 1.000]","0.960 [0.864, 1.000]","0.920 [0.718, 1.000]",0.000,"0.996 [0.986, 0.999]"


In [ ]:
metrics_multi, metricas_multi_raw = display_table("multi")

,f1_macro,precision,recall,mcc,block_rate,tara@5
model,,,,,,
distilbert-base-spanish-uncased,"0.818 [0.758, 0.865]","0.844 [0.778, 0.884]","0.834 [0.775, 0.875]","0.744 [0.673, 0.807]",-,-
roberta-large-bne,"0.842 [0.778, 0.890]","0.846 [0.774, 0.895]","0.845 [0.780, 0.888]","0.786 [0.703, 0.846]",-,-
mdeberta-v3-base,"0.790 [0.725, 0.846]","0.822 [0.762, 0.867]","0.779 [0.705, 0.838]","0.709 [0.628, 0.779]",-,-
llama-3.1-8b,"0.748 [0.684, 0.809]","0.752 [0.692, 0.807]","0.799 [0.739, 0.846]","0.695 [0.621, 0.767]",0.001,"0.837 [0.824, 0.849]"
gpt-oss-20b,"0.724 [0.656, 0.790]","0.734 [0.666, 0.801]","0.752 [0.681, 0.809]","0.655 [0.571, 0.730]",0.000,"0.847 [0.835, 0.859]"
deepseek-chat,"0.766 [0.695, 0.826]","0.796 [0.728, 0.848]","0.760 [0.684, 0.820]","0.684 [0.594, 0.755]",0.000,"0.883 [0.872, 0.894]"
gpt-5.2,"0.790 [0.728, 0.846]","0.809 [0.743, 0.861]","0.820 [0.776, 0.863]","0.728 [0.651, 0.791]",0.000,"0.868 [0.856, 0.878]"
gpt-5-mini,"0.783 [0.719, 0.844]","0.793 [0.726, 0.851]","0.824 [0.772, 0.864]","0.735 [0.653, 0.797]",0.000,"0.863 [0.851, 0.874]"
claude-sonnet-4-6,"0.782 [0.714, 0.838]","0.797 [0.733, 0.851]","0.830 [0.777, 0.868]","0.745 [0.665, 0.802]",0.000,"0.878 [0.867, 0.889]"


## 2.5. Save final table with all metrics

In [ ]:
metricas_anx_raw.to_csv(f"{CACHE_DIR}/metrics_anx.csv")
metricas_dep_raw.to_csv(f"{CACHE_DIR}/metrics_dep.csv")
metricas_ed_raw.to_csv(f"{CACHE_DIR}/metrics_ed.csv")
metricas_multi_raw.to_csv(f"{CACHE_DIR}/metrics_multi.csv")